## OWASPSynthesizer Example

This notebook demonstrates how to use the `OWASPSynthesizer` to generate red-team test cases driven by official OWASP Top 10 report content:
- Automatically downloads and parses the OWASP PDF
- Splits the report into per-risk sections
- Generates adversarial prompts grounded in the official report text

The `OWASPSynthesizer` works with any OWASP Top 10 report:
- **LLM Top 10** (default): `DEFAULT_OWASP_LLM_PDF_URL`
- **Agentic Top 10**: `DEFAULT_OWASP_AGENTIC_PDF_URL`
- **Custom URL**: any PDF following the standard one-section-per-page layout

Key parameters:
- `purpose`: Description of the system under test — used to tailor attack prompts
- `report_url`: URL of the OWASP PDF (defaults to LLM Top 10)
- `categories`: Optional list of section IDs to restrict generation (e.g. `["llm01", "llm07"]`)
- `batch_size`: Maximum tests per LLM call (default: 10)
- `model`: The model to use for generation

In [ ]:
import os

from rhesis.sdk.models.providers.polyphemus import PolyphemusLLM
from rhesis.sdk.synthesizers import (
    DEFAULT_OWASP_AGENTIC_PDF_URL,
    DEFAULT_OWASP_LLM_PDF_URL,
    OWASPSynthesizer,
)

os.environ["RHESIS_API_KEY"] = "replace-with-your-api-key"

POLYPHEMUS_BASE_URL = "https://dev-polyphemus.rhesis.ai"

print("\u2713 SDK configured successfully")
print("Ready to generate OWASP-grounded red-team tests!")

### Example 1: LLM Top 10 — All Sections

In [ ]:
model = PolyphemusLLM(base_url=POLYPHEMUS_BASE_URL)

synthesizer = OWASPSynthesizer(
    purpose=(
        "Customer service chatbot for a retail bank. "
        "Has access to account balances, transaction history, and can initiate transfers."
    ),
    model=model,
    batch_size=5,
    # report_url defaults to DEFAULT_OWASP_LLM_PDF_URL
)

# Tests are distributed evenly across all 10 sections
result = synthesizer.generate(num_tests=10)

print(f"Generated {len(result.tests)} tests")
print(f"Synthesizer: {result.metadata.get('synthesizer')}")
print(f"Report URL: {synthesizer.report_url}")

In [ ]:
# Inspect generated tests
for test in result.tests:
    category = test.metadata.get("owasp_category", "?") if test.metadata else "?"
    name = test.metadata.get("owasp_name", "?") if test.metadata else "?"
    print(f"[{category.upper()}] {name}")
    print(f"  {test.prompt.content[:120]}")
    print()

### Example 2: Agentic Top 10

In [ ]:
model = PolyphemusLLM(base_url=POLYPHEMUS_BASE_URL)

synthesizer = OWASPSynthesizer(
    purpose=(
        "Autonomous coding agent with shell access and filesystem permissions. "
        "Can read/write files, run shell commands, and call external APIs."
    ),
    report_url=DEFAULT_OWASP_AGENTIC_PDF_URL,
    model=model,
    batch_size=5,
)

result = synthesizer.generate(num_tests=10)

print(f"Generated {len(result.tests)} tests from Agentic Top 10")
for test in result.tests:
    category = test.metadata.get("owasp_category", "?") if test.metadata else "?"
    print(f"[{category.upper()}] {test.prompt.content[:100]}")

### Example 3: Filtered to Specific Sections

In [ ]:
model = PolyphemusLLM(base_url=POLYPHEMUS_BASE_URL)

# Target only Prompt Injection (llm01) and System Prompt Leakage (llm07)
synthesizer = OWASPSynthesizer(
    purpose="Legal document summarisation assistant with access to confidential case files.",
    categories=["llm01", "llm07"],
    model=model,
    batch_size=5,
)

result = synthesizer.generate(num_tests=6)

print(f"Generated {len(result.tests)} tests for llm01 + llm07")
for test in result.tests:
    category = test.metadata.get("owasp_category", "?") if test.metadata else "?"
    name = test.metadata.get("owasp_name", "?") if test.metadata else "?"
    print(f"[{category.upper()}] {name}")
    print(f"  {test.prompt.content[:120]}")
    print()